In [10]:
""" Create the observational data set after controlling for pretreatment effects
    "extract_obs_productivity.ipynb" only applies it on biomass

    Because of the considerable pre-treatment effect reported in Hanson et al. 2025,
    we also remove the pre-treatment effect from AGNPP of the trees

    Use simple division!

    Hanson, P. J., Griffiths, N. A., Salmon, V. G., Birkebak, J. M., Warren, J. M., Phillips, J. R., et al. (2025). Peatland Plant Community Changes in Annual Production and Composition Through 8 Years of Warming Manipulations Under Ambient and Elevated CO2 Atmospheres. Journal of Geophysical Research: Biogeosciences, 130(2), e2024JG008511. https://doi.org/10.1029/2024JG008511
"""
import pandas as pd
import os
import numpy as np
import statsmodels.formula.api as smf

In [11]:
obs_paul = pd.read_excel(os.path.join(os.environ['HOME'],
    'Git', 'phenology_elm', "SPRUCE C Budget Summary 28Apr2022EXP.xlsx"),
    sheet_name="DataForPythonRead", skiprows=1,  engine="openpyxl")
obs_paul = obs_paul.loc[obs_paul["Year"] != 2020, :]
obs_paul = obs_paul.set_index(['Plot', 'Year']).sort_index()

obs_verity = pd.read_csv(os.path.join(os.environ['HOME'],
    'Git', 'phenology_elm', 'SalmonSPRUCE_2016to2021_AbovegroundPFT_CNPbudget_20240208.csv'))
# match by plot and year to temperature
obs_verity['Plot'] = [f'P{p:02d}' for p in obs_verity['Plot']]
obs_verity = obs_verity.set_index(['Plot', 'Year', 'PFT']).unstack()
obs_verity = obs_verity.loc[:, (slice(None), 
                                ['Sphagnum', 'evergreen conifer', 'deciduous conifer',  
                                    'shrub'])]
obs_verity2 = obs_verity.loc[:, ['ABGbiomass_gCperm2', 'ABGnpp_gCperm2peryear',
                                 'Pretrt_ABGbiomass_gCperm2', 'Pretrt_ABGnpp_gCperm2peryear']]
obs_data = pd.concat([obs_paul, obs_verity2], axis = 1, join = 'outer')
obs_data = obs_data.reset_index()

# Merge 0 and Amb
obs_data['eCO2'] = np.where(obs_data['CO2'].isna(), np.nan, obs_data['CO2'] == 500)

In [12]:
obs_data2 = pd.DataFrame(np.nan, 
    index = pd.MultiIndex.from_frame(obs_data[['Plot','Year']]),
    columns = ['eCO2', 'Tair', 
               'AGBiomass_Spruce', 'AGBiomass_Tamarack', 'AGBiomass_Shrub', # gC m-2
               'AGNPPtoBiomass_Spruce', 'AGNPPtoBiomass_Tamarack', 'AGNPPtoBiomass_Shrub', # NPP to Biomass ratio, yr-1
               'AGNPP_Spruce', 'AGNPP_Tamarack', 'AGNPP_Shrub', 'NPP_moss' # gC m-2 yr-1
               'BGNPP_TreeShrub', # fine root NPP, gC m-2 yr-1
               'BGtoAG_TreeShrub', # fine root NPP to AGNPP ratio
               'AGNPP', # tree + shrub + moss
               'HR', 'NEE']
)

In [13]:
obs_data2.loc[:, 'eCO2'] = obs_data['eCO2'].values
obs_data2.loc[:, 'Tair'] = obs_data.loc[:, 'Mean Annual Temp. at 2 m'].values
obs_data2.loc[:, 'BGNPP_TreeShrub'] = obs_data['BNPP Tree & Shrub'].values
obs_data2.loc[:, 'HR'] = obs_data.loc[:, 'RHCO2'].values
obs_data2.loc[:, 'NEE'] = obs_data.loc[:, 'NCE'].values
obs_data2.loc[:, 'NPP_moss'] = obs_data.loc[:, 'NPP Sphag.'].values

In [14]:
# Use linear regression to remove pre-treatment effect on the
# aboveground biomass & aboveground NPP of trees
# 
# Post treatment level ~ I(eCO2) + beta1 * Tair
#    + beta2 * Pre treatment level + beta3 * I(eCO2) x Tair
#    + beta4 * I(eCO2) x Pre treatment level + beta 5 * Tair x Pre treatment level
# 
# Subtract out all three terms that include pretreatment effect if significant

def create_X(obs_data, varpre, varname, pft2):
    temp_df = {
        'eCO2': obs_data['eCO2'],
        'Tair': obs_data['Mean Annual Temp. at 2 m'].values,
        'Pretrt': obs_data[(varpre,pft2)].values,
        'Postrt': obs_data[(varname,pft2)].values,
    }
    temp_df = pd.DataFrame(temp_df)
    temp_df = temp_df.dropna(how = 'any', axis = 0)

    return temp_df

for varname, varpre, varfinal in zip(['ABGbiomass_gCperm2', 'ABGnpp_gCperm2peryear'],
                                     ['Pretrt_ABGbiomass_gCperm2', 'Pretrt_ABGnpp_gCperm2peryear'],
                                     ['AGBiomass', 'AGNPP']):
    for pft,pft2 in zip(['Spruce', 'Tamarack', 'Shrub'],
                        ['evergreen conifer','deciduous conifer','shrub']):

        if pft2 == 'shrub':
            obs_data2.loc[:, f'{varfinal}_{pft}'] = obs_data[(varname, pft2)].values
        else:
            obs_data2.loc[:, f'{varfinal}_{pft}'] = obs_data[(varname, pft2)].values / \
                obs_data[(varpre, pft2)].values * np.nanmean(obs_data[(varpre, pft2)].values)

In [15]:
for pft,pft2 in zip(['Spruce', 'Tamarack', 'Shrub'], 
                    ['evergreen conifer','deciduous conifer','shrub']):
    obs_data2.loc[:, f'AGNPPtoBiomass_{pft}'] = \
        obs_data[('ABGnpp_gCperm2peryear', pft2)].values / \
        obs_data[('ABGbiomass_gCperm2', pft2)].values

In [16]:
obs_data2.loc[:, 'BGtoAG_TreeShrub'] = \
    obs_data2.loc[:, 'BGNPP_TreeShrub'].values / \
    (obs_data2.loc[:, 'AGNPP_Spruce'] + obs_data2.loc[:, 'AGNPP_Tamarack'] + \
     obs_data2.loc[:, 'AGNPP_Shrub']).values

In [17]:
obs_data2.loc[:, 'NPP'] = \
    (obs_data2.loc[:, 'AGNPP_Spruce'] + obs_data2.loc[:, 'AGNPP_Tamarack'] + \
     obs_data2.loc[:, 'AGNPP_Shrub'] + obs_data2.loc[:, 'BGNPP_TreeShrub'] + \
     obs_data2.loc[:, 'NPP_moss']).values

In [18]:
path_out = os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 'extract')
obs_data2.to_csv(os.path.join(path_out, 'extract_obs_productivity.csv'))

obs_data2

eCO2       Tair  AGBiomass_Spruce  AGBiomass_Tamarack  \
Plot Year                                                          
P04  2016   1.0  12.700000        942.617508          209.011162   
     2017   1.0  11.300000       1077.046263          228.560999   
     2018   1.0  10.800000       1253.798973          246.134134   
     2019   1.0   9.967064       1376.697482          271.896306   
     2021   1.0  12.100000       1516.507681          258.017498   
...         ...        ...               ...                 ...   
P13  2020   NaN        NaN       1112.837676          346.116745   
P16  2020   NaN        NaN       1041.203769          217.737009   
P17  2020   NaN        NaN        690.120813          260.213311   
P19  2020   NaN        NaN       1022.718967          363.447744   
P20  2020   NaN        NaN       1192.485471          271.106355   

           AGBiomass_Shrub  AGNPPtoBiomass_Spruce  AGNPPtoBiomass_Tamarack  \
Plot Year                                                                    
P04  2016          354.660               0.161327                 0.007305   
     2017          418.203               0.099601                 0.109131   
     2018          273.278               0.150335                 0.145624   
     2019          311.169               0.121375                 0.165687   
     2021          423.661               0.128676                 0.108055   
...                    ...                    ...                      ...   
P13  2020              NaN               0.104109                 0.129768   
P16  2020              NaN               0.070416                 0.139325   
P17  2020              NaN               0.076110                 0.194170   
P19  2020              NaN               0.062666                 0.103896   
P20  2020              NaN               0.099630                 0.152179   

           AGNPPtoBiomass_Shrub  AGNPP_Spruce  AGNPP_Tamarack  AGNPP_Shrub  \
Plot Year                                                                    
P04  2016              0.499774    116.495002        1.170583      177.250   
     2017              0.390753     82.179271       19.122340      163.414   
     2018              0.343566    144.394714       27.478696       93.889   
     2019              0.406605    128.006679       34.536847      126.523   
     2021              0.354991    149.487325       21.373916      150.396   
...                         ...           ...             ...          ...   
P13  2020                   NaN    115.055831       50.171740          NaN   
P16  2020                   NaN     57.803353       39.086253          NaN   
P17  2020                   NaN     45.783978       36.871522          NaN   
P19  2020                   NaN     64.394063       17.871585          NaN   
P20  2020                   NaN    135.283266       43.003420          NaN   

           NPP_mossBGNPP_TreeShrub  BGtoAG_TreeShrub  AGNPP         HR  \
Plot Year                                                                
P04  2016                      NaN          0.355695    NaN -537.30000   
     2017                      NaN          0.411007    NaN -453.30000   
     2018                      NaN          0.402051    NaN -456.80000   
     2019                      NaN          0.373011    NaN -381.07052   
     2021                      NaN          0.334876    NaN -453.75570   
...                            ...               ...    ...        ...   
P13  2020                      NaN               NaN    NaN        NaN   
P16  2020                      NaN               NaN    NaN        NaN   
P17  2020                      NaN               NaN    NaN        NaN   
P19  2020                      NaN               NaN    NaN        NaN   
P20  2020                      NaN               NaN    NaN        NaN   

                  NEE  BGNPP_TreeShrub  NPP_moss         NPP  
Plot Year                                                     
P04  20

In [24]:
pretreatment = obs_verity.loc[(slice(None), 2016), 'ABGnpp_gCperm2peryear']
print('mean', pretreatment.mean())
print('std', pretreatment.std())

mean PFT
Sphagnum             111.2469
evergreen conifer     90.4510
deciduous conifer     23.3357
shrub                 92.1004
dtype: float64
std PFT
Sphagnum             21.078571
evergreen conifer    53.490134
deciduous conifer    15.905067
shrub                34.932913
dtype: float64
